# E3′ — BERTimbau valida o FALCO (Colab / Databricks)

**O que este notebook faz:** treina o BERTimbau (classificador forte) em 5 braços
e mede na população reservada se o pipeline barato (laço leve + oráculo LLM)
entrega um modelo forte competitivo. Cada braço é UM ajuste fino — o laço já
rodou; os conjuntos estão prontos no repositório.

| Braço | Treino | Responde |
|---|---|---|
| A | 8.937 itens anotados pelo pipeline real (oráculo NIM) | o que o pipeline entrega |
| B | mesmos itens, rótulos gold | custo do ruído do oráculo (A−B) |
| C | 8.937 aleatórios, gold | valor da seleção (B−C) |
| D | pool 50k, gold | a régua (teto do pool) |
| E | 15k por entropia (E6), gold | orçamento maior ajudaria? |

**Hipótese:** F1(A) ≥ 0,95 × F1(D) com ~18% dos rótulos.

## O que você precisa
1. **GPU**: em *Runtime → Change runtime type*, escolha **T4 GPU** (Colab). 
2. **Acesso ao repositório** `GHDaru/activelearning`: um *fine-grained token* do
   GitHub com leitura de conteúdo (cole quando o passo 2 pedir). 
3. **`data/dataset.csv`**: se o repositório não contiver a base, envie o CSV no
   passo 3 (upload ou Google Drive). 

Tempo estimado numa T4: **~40–70 min** os 5 braços com avaliação na população
inteira (177k). Os resultados ficam em `experiments/e2e3/results/` e o passo 6
baixa um zip — reenvie esse zip para o repositório (ou para o Claude) ao final.


In [ ]:
# 1) GPU disponível?
!nvidia-smi -L || echo 'SEM GPU: mude o runtime para T4 (Colab) ou use cluster GPU (Databricks)'


In [ ]:
# 2) Clonar o repositório (cole seu token quando pedido)
import os
from getpass import getpass
if not os.path.exists('activelearning'):
    tok = getpass('GitHub token (fine-grained, leitura de GHDaru/activelearning): ')
    !git clone https://{tok}@github.com/GHDaru/activelearning.git
%cd activelearning


In [ ]:
# 3) Base de dados — pula se data/dataset.csv já veio no clone
import os
if not os.path.exists('data/dataset.csv'):
    print('Envie o dataset.csv (colunas nm_item, nm_product):')
    from google.colab import files          # em Databricks: use dbfs/Volumes e copie p/ data/
    up = files.upload()
    os.makedirs('data', exist_ok=True)
    fname = next(iter(up))
    os.replace(fname, 'data/dataset.csv')
!wc -l data/dataset.csv


In [ ]:
# 4) Dependências (Colab já traz torch com CUDA; só falta transformers)
%pip -q install transformers scikit-learn
import torch; print('CUDA:', torch.cuda.is_available())


In [ ]:
# 5) Rodar os 5 braços (retomada automática: braço concluído é pulado —
#    se a sessão cair, basta reexecutar esta célula)
!python experiments/e2e3/run_e3prime.py --arms A,B,C,E,D \
    --epochs 3 --batch-size 128 --eval-limit 0


In [ ]:
# 6) Consolidar, exibir e baixar os resultados
import json, glob, pandas as pd
rows = [json.load(open(p)) for p in sorted(glob.glob('experiments/e2e3/results/e3prime_?.json'))]
df = pd.DataFrame(rows)[['arm','n_train','n_train_classes','accuracy','accuracy_wilson95','macro_f1','fit_seconds']]
display(df)
if {'A','D'} <= set(df.arm):
    fa = float(df[df.arm=='A'].macro_f1.iloc[0]); fd = float(df[df.arm=='D'].macro_f1.iloc[0])
    na = int(df[df.arm=='A'].n_train.iloc[0]);  nd = int(df[df.arm=='D'].n_train.iloc[0])
    ok = fa >= 0.95*fd
    print(f'HIPÓTESE: F1(A)={fa:.4f} vs 0,95×F1(D)={0.95*fd:.4f} com {na/nd:.1%} dos rótulos ->', 'SUSTENTADA' if ok else 'NÃO sustentada')
!zip -qr e3prime_results.zip experiments/e2e3/results
try:
    from google.colab import files; files.download('e3prime_results.zip')
except ImportError:
    print('Databricks: baixe experiments/e2e3/results/ pelo painel de arquivos ou copie para um Volume')


## Databricks (em vez do Colab)
1. **Cluster**: single node com GPU (ex.: `g4dn.xlarge`/T4 na AWS ou `Standard_NC4as_T4_v3`
   no Azure), runtime *ML com GPU* (torch já incluso). 
2. **Código**: *Repos → Add Repo* → `https://github.com/GHDaru/activelearning` (conecte sua
   conta GitHub) — dispensa o passo 2. 
3. **Dados**: suba `dataset.csv` para um Volume (`/Volumes/<catalog>/<schema>/<vol>/dataset.csv`)
   e copie: `%sh cp /Volumes/.../dataset.csv data/dataset.csv`. 
4. **Dependências**: `%pip install transformers scikit-learn`. 
5. **Execução**: as células 5–6 funcionam iguais (o `files.download` não existe —
   os resultados ficam no Repo/Volume). 

## Alternativa sem notebook
A imagem Docker do experimento roda os mesmos comandos em qualquer máquina com GPU
(vast.ai/RunPod cobram ~US$ 0,20–0,40/h por uma 3090/4090): ver `docs/e2e3-docker.md`.
